In [1]:
from pyspark.sql import SparkSession

In [17]:
#importing the necessary libraries
from pyspark.sql.functions import countDistinct,sum,col,lit, when
from pyspark.sql import Row
from pyspark.sql.types import *

In [3]:
#building the spark session

spark = SparkSession.builder.appName("Crash Analysis").getOrCreate()
print(spark)

25/06/11 12:18:36 WARN Utils: Your hostname, TTNPL-8203 resolves to a loopback address: 127.0.1.1; using 10.1.209.119 instead (on interface wlp0s20f3)
25/06/11 12:18:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/11 12:18:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### defining the Schema and reading the data

In [4]:
drivers_schema = StructType([
    StructField("Report Number",StringType(),True),
    StructField("Local Case Number",IntegerType(),True),
     StructField("Agency Name", StringType(), True),
    StructField("ACRS Report Type", StringType(), True),
    StructField("Crash Date/Time", TimestampType(), True),  # Assuming proper datetime format
    StructField("Route Type", StringType(), True),
    StructField("Road Name", StringType(), True),
    StructField("Cross-Street Name", StringType(), True),
    StructField("Off-Road Description", StringType(), True),
    StructField("Municipality", StringType(), True),
    StructField("Related Non-Motorist", StringType(), True),
    StructField("Collision Type", StringType(), True),
    StructField("Weather", StringType(), True),
    StructField("Surface Condition", StringType(), True),
    StructField("Light", StringType(), True),
    StructField("Traffic Control", StringType(), True),
    StructField("Driver Substance Abuse", StringType(), True),
    StructField("Non-Motorist Substance Abuse", StringType(), True),
    StructField("Person ID", StringType(), True),
    StructField("Driver At Fault", StringType(), True),
    StructField("Injury Severity", StringType(), True),
    StructField("Circumstance", StringType(), True),
    StructField("Driver Distracted By", StringType(), True),
    StructField("Drivers License State", StringType(), True),
    StructField("Vehicle ID", StringType(), True),
    StructField("Vehicle Damage Extent", StringType(), True),
    StructField("Vehicle First Impact Location", StringType(), True),
    StructField("Vehicle Body Type", StringType(), True),
    StructField("Vehicle Movement", StringType(), True),
    StructField("Vehicle Going Dir", StringType(), True),
    StructField("Speed Limit", IntegerType(), True),
    StructField("Driverless Vehicle", StringType(), True),
    StructField("Parked Vehicle", StringType(), True),
    StructField("Vehicle Year", IntegerType(), True),
    StructField("Vehicle Make", StringType(), True),
    StructField("Vehicle Model", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True)
    
])

In [5]:
drivers_df = spark.read.csv("file:///home/hdoop/notebooks/capstone-project-2/data/drivers_data/Crash_Reporting_-_Drivers_Data_20250610.csv",header =True,schema=drivers_schema,timestampFormat="MM/dd/yyyy hh:mm:ss a")


In [6]:
#incidents data schema

incidents_schema = StructType([
    StructField("Report Number",StringType(),True),
    StructField("Local Case Number",IntegerType(),True),
    StructField("Agency Name",StringType(),True),
    StructField("ACRS Report Type",StringType(),True),
    StructField("Crash Date/Time",TimestampType(),True),
    StructField("Hit/Run",StringType(),True),
    StructField("Route Type",StringType(),True),
    StructField("Lane Direction",StringType(),True),
    StructField("Lane Type",StringType(),True),
    StructField("Number of Lanes",StringType(),True),
    StructField("Direction",StringType(),True),
    StructField("Distance",DoubleType(),True),
    StructField("Distance Unit",StringType(),True),
    StructField("Road Grade",StringType(),True),
    StructField("Cross-Street Name",StringType(),True),
    StructField("Off-Road Description",StringType(),True),
    StructField("Municipality",StringType(),True),
    StructField("Related Non-Motorist",StringType(),True),
    StructField("At Fault",StringType(),True),
    StructField("Collision Type",StringType(),True),
    StructField("Weather",StringType(),True),
    StructField("Surface Condition",StringType(),True),
    StructField("Light",StringType(),True),
    StructField("Traffic Control",StringType(),True),
    StructField("Driver Substance Abuse",StringType(),True),
    StructField("Non-Motorist Substance Abuse",StringType(),True),
    StructField("First Harmful Event",StringType(),True),
    StructField("Second Harmful Event",StringType(),True),
    StructField("Junction",StringType(),True),
    StructField("Intersection Type",StringType(),True),
    StructField("Road Alignment",StringType(),True),
    StructField("Road Condition",StringType(),True),
    StructField("Road Division",StringType(),True),
    StructField("Latitude",DoubleType(),True),
    StructField("Logitude",DoubleType(),True),
    StructField("Location",StringType(),True)
    
])

In [7]:
incidents_df = spark.read.csv("file:///home/hdoop/notebooks/capstone-project-2/data/incidents_data/Crash_Reporting_-_Incidents_Data_20250610.csv",header=True,schema = incidents_schema,timestampFormat="MM/dd/yyyy hh:mm:ss a" )

In [8]:
# Non-motoirsts data

non_motorists_schema = StructType([
    StructField("Report Number", StringType(), True),
    StructField("Local Case Number", IntegerType(), True),
    StructField("Agency Name", StringType(), True),
    StructField("ACRS Report Type", StringType(), True),
    StructField("Crash Date/Time", TimestampType(), True),  # Can be changed to TimestampType if parsed with timestampFormat
    StructField("Route Type", StringType(), True),
    StructField("Road Name", StringType(), True),
    StructField("Cross-Street Name", StringType(), True),
    StructField("Off-Road Description", StringType(), True),
    StructField("Municipality", StringType(), True),
    StructField("Related Non-Motorist", StringType(), True),
    StructField("Collision Type", StringType(), True),
    StructField("Weather", StringType(), True),
    StructField("Surface Condition", StringType(), True),
    StructField("Light", StringType(), True),
    StructField("Traffic Control", StringType(), True),
    StructField("Driver Substance Abuse", StringType(), True),
    StructField("Non-Motorist Substance Abuse", StringType(), True),
    StructField("Person ID", StringType(), True),
    StructField("Pedestrian Type", StringType(), True),
    StructField("Pedestrian Movement", StringType(), True),
    StructField("Pedestrian Actions", StringType(), True),
    StructField("Pedestrian Location", StringType(), True),
    StructField("At Fault", StringType(), True),
    StructField("Injury Severity", StringType(), True),
    StructField("Safety Equipment", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True)  # parsed later if needed
])

In [9]:

non_motorists_df = spark.read.csv("file:///home/hdoop/notebooks/capstone-project-2/data/non-motorists_data/Crash_Reporting_-_Non-Motorists_Data_20250610.csv",header = True,schema = non_motorists_schema,timestampFormat="MM/dd/yyyy hh:mm:ss a")

### Analysis on the data

In [13]:
# drivers_summary_data = drivers_df.select([
#     sum(col(c).isNull().cast("int")).alias(f"{c}_nulls")
#     for c in drivers_df.columns
# ] + [
#     countDistinct(col(c)).alias(f"{c}_uniques")
#     for c in drivers_df.columns
# ])

# drivers_summary_data.show(vertical = True,truncate = False)

25/06/11 12:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:28:22 WARN RowBasedKeyValueBatch: Calling spill() on

-RECORD 0---------------------------------------
 Report Number_nulls                   | 0      
 Local Case Number_nulls               | 1528   
 Agency Name_nulls                     | 1371   
 ACRS Report Type_nulls                | 727    
 Crash Date/Time_nulls                 | 1476   
 Route Type_nulls                      | 20400  
 Road Name_nulls                       | 22284  
 Cross-Street Name_nulls               | 32600  
 Off-Road Description_nulls            | 179884 
 Municipality_nulls                    | 42623  
 Related Non-Motorist_nulls            | 192161 
 Collision Type_nulls                  | 1524   
 Weather_nulls                         | 1494   
 Surface Condition_nulls               | 18799  
 Light_nulls                           | 1504   
 Traffic Control_nulls                 | 3794   
 Driver Substance Abuse_nulls          | 1536   
 Non-Motorist Substance Abuse_nulls    | 191838 
 Person ID_nulls                       | 1479   
 Driver At Fault_nul

In [20]:
# 1. Total number of rows (cached for reuse)
total_rows = drivers_df.count()

# 2. Compute nulls and uniques efficiently
summary_df = drivers_df.select([
    count(when(col(c).isNull(), c)).alias(f"{c}_nulls") for c in drivers_df.columns
] + [
    countDistinct(col(c)).alias(f"{c}_uniques") for c in drivers_df.columns
])

# 3. Transpose results to have one row per column
summary_data = []
row = summary_df.first()

for c in drivers_df.columns:
    summary_data.append((c, row[f"{c}_nulls"], row[f"{c}_uniques"], total_rows))

# 4. Convert to a nice summary DataFrame
from pyspark.sql import Row
summary_result = spark.createDataFrame(
    [Row(column=col_name, nulls=nulls, uniques=uniques, total=total) 
     for col_name, nulls, uniques, total in summary_data]
)

# 5. Show final result
summary_result.orderBy("nulls", ascending=False).show(truncate=False)

25/06/11 12:43:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:43:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:43:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:43:50 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:43:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:43:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:43:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:43:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:43:51 WARN RowBasedKeyValueBatch: Calling spill() on

+----------------------------+------+-------+------+
|column                      |nulls |uniques|total |
+----------------------------+------+-------+------+
|Related Non-Motorist        |192161|760    |199275|
|Non-Motorist Substance Abuse|191838|402    |199275|
|Off-Road Description        |179884|12872  |199275|
|Municipality                |42623 |37     |199275|
|Cross-Street Name           |32600 |7400   |199275|
|Road Name                   |22284 |4573   |199275|
|Circumstance                |20707 |741    |199275|
|Route Type                  |20400 |34     |199275|
|Surface Condition           |18799 |56     |199275|
|Drivers License State       |14176 |98     |199275|
|Vehicle Going Dir           |8544  |852    |199275|
|Vehicle Body Type           |4220  |421    |199275|
|Parked Vehicle              |4086  |44     |199275|
|Traffic Control             |3794  |66     |199275|
|Vehicle Model               |3305  |7051   |199275|
|Vehicle Make                |3302  |1950   |1

In [21]:
summary_result.orderBy("nulls", ascending=False).show(n=100,truncate=False)

+-----------------------------+------+-------+------+
|column                       |nulls |uniques|total |
+-----------------------------+------+-------+------+
|Related Non-Motorist         |192161|760    |199275|
|Non-Motorist Substance Abuse |191838|402    |199275|
|Off-Road Description         |179884|12872  |199275|
|Municipality                 |42623 |37     |199275|
|Cross-Street Name            |32600 |7400   |199275|
|Road Name                    |22284 |4573   |199275|
|Circumstance                 |20707 |741    |199275|
|Route Type                   |20400 |34     |199275|
|Surface Condition            |18799 |56     |199275|
|Drivers License State        |14176 |98     |199275|
|Vehicle Going Dir            |8544  |852    |199275|
|Vehicle Body Type            |4220  |421    |199275|
|Parked Vehicle               |4086  |44     |199275|
|Traffic Control              |3794  |66     |199275|
|Vehicle Model                |3305  |7051   |199275|
|Vehicle Make               

In [22]:
# 1. Total number of rows (cached for reuse)
total_rows = incidents_df.count()

# 2. Compute nulls and uniques efficiently
summary_df = incidents_df.select([
    count(when(col(c).isNull(), c)).alias(f"{c}_nulls") for c in incidents_df.columns
] + [
    countDistinct(col(c)).alias(f"{c}_uniques") for c in incidents_df.columns
])

# 3. Transpose results to have one row per column
summary_data = []
row = summary_df.first()

for c in incidents_df.columns:
    summary_data.append((c, row[f"{c}_nulls"], row[f"{c}_uniques"], total_rows))

# 4. Convert to a nice summary DataFrame
from pyspark.sql import Row
summary_result = spark.createDataFrame(
    [Row(column=col_name, nulls=nulls, uniques=uniques, total=total) 
     for col_name, nulls, uniques, total in summary_data]
)

# 5. Show final result
summary_result.orderBy("nulls", ascending=False).show(truncate=False,n=100)

25/06/11 12:46:48 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 37, schema size: 36
CSV file: file:///home/hdoop/notebooks/capstone-project-2/data/incidents_data/Crash_Reporting_-_Incidents_Data_20250610.csv
25/06/11 12:46:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:46:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/06/11 12:46:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

+----------------------------+------+-------+------+
|column                      |nulls |uniques|total |
+----------------------------+------+-------+------+
|Latitude                    |113313|0      |113313|
|First Harmful Event         |107087|28     |113313|
|At Fault                    |106545|576    |113313|
|Municipality                |99175 |13145  |113313|
|Lane Type                   |89069 |187    |113313|
|Related Non-Motorist        |28168 |27     |113313|
|Road Alignment              |24087 |15     |113313|
|Off-Road Description        |23239 |7416   |113313|
|Road Division               |17332 |22     |113313|
|Cross-Street Name           |17088 |4601   |113313|
|Intersection Type           |15713 |25     |113313|
|Route Type                  |15607 |108    |113313|
|Road Condition              |15392 |15     |113313|
|Light                       |15358 |61     |113313|
|Road Grade                  |15025 |36     |113313|
|Direction                   |14979 |23     |1

In [ ]:
drivers_df.where(col("Report"))